# 02. ROSE and LROM Cross-Section Comparison

In [ ]:
from pathlib import Path
import platform
import sys
import time

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from numba import njit

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "lrom_legacy").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scipy.special
if not hasattr(scipy.special, "sph_harm") and hasattr(scipy.special, "sph_harm_y"):
    scipy.special.sph_harm = (
        lambda m, n, theta, phi: scipy.special.sph_harm_y(n, m, phi, theta)
    )

import rose
import lrom_legacy.v2_0 as lrom

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
print("lrom version:", lrom.__version__)


## 1. Physical Problem and Optical-Potential Parameters

In [ ]:
TARGET = (40, 20)
PROJECTILE = (1, 0)
LAB_ENERGY = 14.1
L_MAX = 3
MESH_SIZE = 600
N_TRAIN = 200
N_TEST = 100
HALF_WIDTH = 0.20
ANGLES_DEG = np.arange(1.0, 180.0, 1.0)
ANGLES_RAD = np.deg2rad(ANGLES_DEG)
BASIS_SIZES = (4, 6, 8)
ROSE_EIM_SIZES = (4, 8, 12)
LROM_PREDICTOR_COUNTS = (4, 8, 12)
DEFAULT_BASIS_SIZE = 6
DEFAULT_COMPRESSION_SIZE = 8
TIMING_REPEATS = 3
SEED = 1204
ERROR_DENOMINATOR_FLOOR = 1e-12
PLOTTING_FLOOR = 1e-6


## 2. Shared Training and Testing Samples

In [ ]:
emulator = lrom.LROM(
    target=TARGET,
    projectile=PROJECTILE,
    lab_energy=LAB_ENERGY,
    l=tuple(range(L_MAX + 1)),
    potential="full_woods-saxon",
)
central = dict(emulator.central_parameters)
ranges = {
    name: tuple(sorted(((1.0 - HALF_WIDTH) * value, (1.0 + HALF_WIDTH) * value)))
    for name, value in central.items()
}
emulator.sampling(
    training_ranges=ranges,
    testing_ranges=ranges,
    training_size=N_TRAIN,
    testing_size=N_TEST,
    mesh_size=MESH_SIZE,
    strategy="latin_hypercube",
    seed=SEED,
    high_fidelity_solver="runge_kutta",
    solver_options={"rk_tols": (1e-9, 1e-9)},
)
parameter_names = emulator.parameter_names
train_rows = emulator.samples.design.training.values.copy()
test_rows = emulator.samples.design.testing.values.copy()
train_ids = emulator.samples.design.training.case_ids
test_ids = emulator.samples.design.testing.case_ids

assert train_rows.shape == (N_TRAIN, len(parameter_names))
assert test_rows.shape == (N_TEST, len(parameter_names))
assert not any(np.array_equal(a, b) for a in train_rows for b in test_rows)
for column, name in enumerate(parameter_names):
    lower, upper = ranges[name]
    assert np.all((lower <= train_rows[:, column]) & (train_rows[:, column] <= upper))
    assert np.all((lower <= test_rows[:, column]) & (test_rows[:, column] <= upper))

rose_train_rows = train_rows
rose_test_rows = test_rows
assert np.array_equal(train_rows, rose_train_rows)
assert np.array_equal(test_rows, rose_test_rows)


def rows_as_parameter_dicts(rows):
    return [
        {name: float(value) for name, value in zip(parameter_names, row)}
        for row in rows
    ]


def pointwise_relative_error(predicted, reference):
    predicted = np.asarray(predicted, dtype=float)
    reference = np.asarray(reference, dtype=float)
    denominator = np.maximum(np.abs(reference), ERROR_DENOMINATOR_FLOOR)
    return np.abs(predicted - reference) / denominator


def error_summaries(predicted, reference):
    pointwise = pointwise_relative_error(predicted, reference)
    return {
        "median_error": np.median(pointwise, axis=1),
        "cat_error": np.max(pointwise_relative_error(predicted, reference), axis=1),
    }


def minimum_lrom_times(model, cases):
    model.predict(parameters=cases[:1])
    seconds = []
    for case in cases:
        repeats = []
        for _ in range(TIMING_REPEATS):
            start = time.perf_counter()
            model.predict(parameters=case)
            repeats.append(time.perf_counter() - start)
        seconds.append(min(repeats))
    return np.asarray(seconds)


def minimum_rose_times(model, rows):
    model.emulate_dsdo(rows[0])
    seconds = []
    for row in rows:
        repeats = []
        for _ in range(TIMING_REPEATS):
            start = time.perf_counter()
            model.emulate_dsdo(row)
            repeats.append(time.perf_counter() - start)
        seconds.append(min(repeats))
    return np.asarray(seconds)


LS_LABEL = "LS-projected cross section"


## 3. Potential Variation and LROM Predictor Locations

## 4. Equal-Basis ROSE and LROM Emulators

In [ ]:
@njit
def bench_ws(r, radius, diffuseness):
    return 1.0 / (1.0 + np.exp((r - radius) / diffuseness))


@njit
def bench_ws_prime(r, radius, diffuseness):
    ex = np.exp((r - radius) / diffuseness)
    return -(ex / diffuseness) / (1.0 + ex) ** 2


@njit
def bench_full_ws(r, alpha):
    vv, wv, wd, _vso, rv, rd, _rso, av, ad, _aso = alpha
    return (
        -vv * bench_ws(r, rv, av)
        - 1j * wv * bench_ws(r, rv, av)
        + 4j * ad * wd * bench_ws_prime(r, rd, ad)
    )


@njit
def bench_full_ws_so(r, alpha, ldots):
    _vv, _wv, _wd, vso, _rv, _rd, rso, _av, _ad, aso = alpha
    return vso / 139.57039**2 * ldots * bench_ws_prime(r, rso, aso) / r


rho_mesh = emulator.samples.mesh.rho
radius_mesh = emulator.samples.mesh.radius
rose_bounds = np.column_stack(
    [
        np.minimum(train_rows.min(axis=0), test_rows.min(axis=0)),
        np.maximum(train_rows.max(axis=0), test_rows.max(axis=0)),
    ]
)
rose_solver = rose.SchroedingerEquation.make_base_solver(
    s_0=6 * np.pi,
    rk_tols=[1e-9, 1e-9],
    domain=np.array([rho_mesh[0], rho_mesh[-1]]),
)
rose_emulators = {}
for n_phi in BASIS_SIZES:
    for n_u in ROSE_EIM_SIZES:
        interaction = rose.InteractionEIMSpace(
            l_max=L_MAX,
            coordinate_space_potential=bench_full_ws,
            spin_orbit_term=bench_full_ws_so,
            n_theta=len(parameter_names),
            mu=emulator.kinematics.mu,
            energy=emulator.kinematics.e_com,
            is_complex=True,
            training_info=rose_bounds,
            n_basis=n_u,
            rho_mesh=rho_mesh,
        )
        rose_emulators[(n_phi, n_u)] = rose.ScatteringAmplitudeEmulator.from_train(
            interaction,
            rose_train_rows,
            base_solver=rose_solver,
            l_max=L_MAX,
            angles=ANGLES_RAD,
            n_basis=n_phi,
            use_svd=True,
            scale=False,
            s_mesh=rho_mesh,
            Smatrix_abs_tol=1e-8,
        )

reference_sae = rose_emulators[(BASIS_SIZES[0], ROSE_EIM_SIZES[0])]
fom_train_xs = np.asarray([reference_sae.exact_dsdo(row) for row in train_rows])
fom_test_xs = np.asarray([reference_sae.exact_dsdo(row) for row in test_rows])
assert fom_train_xs.shape == (N_TRAIN, ANGLES_DEG.size)
assert fom_test_xs.shape == (N_TEST, ANGLES_DEG.size)
assert np.all(np.isfinite(fom_train_xs)) and np.all(fom_train_xs >= -1e-12)
assert np.all(np.isfinite(fom_test_xs)) and np.all(fom_test_xs >= -1e-12)


train_cases = rows_as_parameter_dicts(train_rows)
test_cases = rows_as_parameter_dicts(test_rows)
lrom_results = {}
ls_results = {}
default_predictor = None

for n_phi in BASIS_SIZES:
    for predictor_count in LROM_PREDICTOR_COUNTS:
        emulator.train(
            basis_size=n_phi,
            predictor="potential",
            predictor_count=predictor_count,
            observable="cross_section",
            angles_degrees=ANGLES_DEG,
        )
        test_seconds = minimum_lrom_times(emulator, test_cases)
        emulator.predict(parameters=train_cases)
        train_xs = emulator.predictions.cross_sections.values.copy()
        emulator.predict(parameters=test_cases)
        test_xs = emulator.predictions.cross_sections.values.copy()
        train_summary = error_summaries(train_xs, fom_train_xs)
        test_summary = error_summaries(test_xs, fom_test_xs)
        lrom_results[(n_phi, predictor_count)] = {
            "train_xs": train_xs,
            "test_xs": test_xs,
            "train_median_error": train_summary["median_error"],
            "test_median_error": test_summary["median_error"],
            "train_cat_error": train_summary["cat_error"],
            "test_cat_error": test_summary["cat_error"],
            "test_seconds": test_seconds,
        }

        if (n_phi, predictor_count) == (
            DEFAULT_BASIS_SIZE,
            DEFAULT_COMPRESSION_SIZE,
        ):
            default_predictor = emulator.predictors

        if predictor_count == LROM_PREDICTOR_COUNTS[0]:
            ls_train_coordinates = {
                channel: lrom.project_coordinates(
                    basis=emulator.basis[channel],
                    wavefunctions=emulator.samples.training_wavefunctions[channel],
                )
                for channel in emulator.basis
            }
            ls_test_coordinates = {
                channel: lrom.project_coordinates(
                    basis=emulator.basis[channel],
                    wavefunctions=emulator.samples.testing_wavefunctions[channel],
                )
                for channel in emulator.basis
            }
            _, ls_train_state = lrom._cross_section_prediction(
                emulator=emulator,
                values=train_rows,
                coefficients=ls_train_coordinates,
            )
            _, ls_test_state = lrom._cross_section_prediction(
                emulator=emulator,
                values=test_rows,
                coefficients=ls_test_coordinates,
            )
            ls_train_summary = error_summaries(ls_train_state.values, fom_train_xs)
            ls_test_summary = error_summaries(ls_test_state.values, fom_test_xs)
            ls_results[n_phi] = {
                "train_xs": ls_train_state.values.copy(),
                "test_xs": ls_test_state.values.copy(),
                "train_median_error": ls_train_summary["median_error"],
                "test_median_error": ls_test_summary["median_error"],
                "train_cat_error": ls_train_summary["cat_error"],
                "test_cat_error": ls_test_summary["cat_error"],
            }

assert default_predictor is not None


rose_results = {}
for config, rose_emulator in rose_emulators.items():
    test_seconds = minimum_rose_times(rose_emulator, test_rows)
    train_xs = np.asarray([rose_emulator.emulate_dsdo(row) for row in train_rows])
    test_xs = np.asarray([rose_emulator.emulate_dsdo(row) for row in test_rows])
    train_summary = error_summaries(train_xs, fom_train_xs)
    test_summary = error_summaries(test_xs, fom_test_xs)
    rose_results[config] = {
        "train_xs": train_xs,
        "test_xs": test_xs,
        "train_median_error": train_summary["median_error"],
        "test_median_error": test_summary["median_error"],
        "train_cat_error": train_summary["cat_error"],
        "test_cat_error": test_summary["cat_error"],
        "test_seconds": test_seconds,
    }

for collection in (lrom_results, rose_results, ls_results):
    for result in collection.values():
        for values in result.values():
            assert np.all(np.isfinite(values))

assert set(lrom_results) == {
    (n_phi, predictor_count)
    for n_phi in BASIS_SIZES
    for predictor_count in LROM_PREDICTOR_COUNTS
}
assert set(rose_results) == {
    (n_phi, n_u)
    for n_phi in BASIS_SIZES
    for n_u in ROSE_EIM_SIZES
}


## 5. Representative Cross-Section Predictions

## 6. Cross-Section Error Distributions

## 7. Basis and Operator-Size Comparison

## 8. Accuracy Versus Online Time

## 9. Validation Summary